In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [2]:
load_dotenv(override=True)
gemini_api_key=os.getenv('GEMINI_API_KEY')
open_router_key=os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GEMINI_BASE_URL="https://generativelanguage.googleapis.com/v1beta/openai/"


In [3]:

if open_router_key:
    print(f"OpenRouter API Key exists and begins {open_router_key[:2]}")
else:
    print("OpenRouter API Key not set")
if gemini_api_key:
    print(f"Google API Key exists and begins {gemini_api_key[:2]}")
else:
    print("Google API Key not set")


OpenRouter API Key exists and begins sk
Google API Key exists and begins AQ


In [4]:
reader=PdfReader("Linkedin.pdf")
linkedin=""
for page in reader.pages:
    text=page.extract_text()
    if text:
        linkedin+=text

In [5]:
print(linkedin)

   
Contact
kotianvibha4@gmail.com
www.linkedin.com/in/vibha-kotian-
b18457209 (LinkedIn)
Top Skills
Python (Programming Language)
PHP
English
vibha kotian
Student at Shreemati Nathibai Damodar Thackersey Women's
University
Vasai Virar, Maharashtra, India
Experience
Bank of America
Apprentice
August 2024 - Present (1 year 11 months)
Tata Institute of Fundamental Research, Mumbai
Project Student
February 2024 - April 2024 (3 months)
Mumbai, Maharashtra, India
Education
Usha Mittal Institute of Technology
Bachelor of Technology - BTech, Information Technology · (December
2021 - May 2024)
Shreemati Nathibai Damodar Thackersey Women's University
Bachelor of Technology - BTech, Information Technology · (2020 - 2023)
  Page 1 of 1


In [6]:
with open("summary.txt",'r',encoding="utf-8")as f:
    summary=f.read()

In [7]:
name="Vibha Kotian"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
gemini=OpenAI(base_url=GEMINI_BASE_URL,api_key=gemini_api_key)
open_router=OpenAI(base_url=OPENROUTER_BASE_URL,api_key=open_router_key)
gemmodel_name = "gemini-2.5-flash"

In [11]:
from urllib3 import response


def chat(message,history):
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    reponse=gemini.chat.completions.create(model=gemmodel_name,messages=messages)
    return reponse.choices[0].message.content

In [12]:
gr.ChatInterface(chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
from pydantic import BaseModel
class Evaluation(BaseModel):
    is_acceptable:bool
    feedback:str

In [14]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [15]:
def evaluator_user_prompt(reply,message,history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [16]:
def evaluate(reply,message,history)->Evaluation:
    open_router_model="openai/gpt-oss-20b:free"
    messages=[{"role":"system","content":evaluator_system_prompt}]+[{"role":"user","content":evaluator_user_prompt(reply,message,history)}]
    response=open_router.beta.chat.completions.parse(model=open_router_model,messages=messages,response_format=Evaluation)
    return response.choices[0].message.parsed

In [17]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response=open_router.chat.completions.create(model="openai/gpt-oss-20b:free",messages=messages)
reply=response.choices[0].message.content


In [18]:
reply

'I don’t have any patents to my name at the moment. I’m focused on applying my programming and web‑development skills across a range of academic projects and real‑world assignments, and I’m open to exploring intellectual property opportunities as my career progresses.'

In [22]:
def rerun(reply,message,history,feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = open_router.chat.completions.create(model="openai/gpt-oss-20b:free", messages=messages)
    return response.choices[0].message.content

In [24]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = open_router.chat.completions.create(model="openai/gpt-oss-20b:free", messages=messages
    
    )
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [25]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply


Traceback (most recent call last):
  File "c:\Users\Vibha\project1\agents\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vibha\project1\agents\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vibha\project1\agents\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vibha\project1\agents\.venv\Lib\site-packages\gradio\blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vibha\project1\agents\.venv\Lib\site-packages\gradio\utils.py", line 882, in async_wrapper
    response = await f(*args, **kwargs)
    